<a href="https://colab.research.google.com/github/bollaprashanth09/prashanth_python/blob/main/Module_7_Project__Smart_Email_Spam_%26_Emotion_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STEP 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install nltk textblob scikit-learn

# Imports
import pandas as pd
import numpy as np
import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

from textblob import TextBlob

STEP 2: Download NLTK Data

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

STEP 3: Upload Dataset

In [ ]:
import pandas as pd

# Re-load the dataset to restore corrupted columns
df = pd.read_csv('/content/spam_email_dataset.csv', encoding='latin1')

print("Dataset reloaded. Total rows:", len(df))
df.head()

Dataset reloaded. Total rows: 1500


,Subject,Body,Spam Label
0,Exclusive Loan Offer  Get Instant Approval,Alert!\n\nYour computer has been infected with...,1
1,Meeting Agenda for Tomorrow,"Dear Student,\n\nThis is a reminder about your...",0
2,Quarterly Sales Report Attached,"Hey,\n\nIm sharing the travel itinerary for o...",0
3,Exclusive Loan Offer  Get Instant Approval,"Dear Investor,\n\nDon't miss this incredible i...",1
4,"Win $1,000,000 Now  Limited Time Offer!","Dear Customer,\n\nWe are giving away free holi...",1


STEP 4: Data Preprocessing (NLP)

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove special chars
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing
df['clean_text'] = df['Body'].apply(preprocess_text)

df[['Body', 'clean_text']].head()

,Body,clean_text
0,Alert!\n\nYour computer has been infected with...,alert computer infected dangerous virus click ...
1,"Dear Student,\n\nThis is a reminder about your...",dear student reminder upcoming online class sc...
2,"Hey,\n\nIm sharing the travel itinerary for o...",hey sharing travel itinerary upcoming trip let...
3,"Dear Investor,\n\nDon't miss this incredible i...",dear investor miss incredible investment oppor...
4,"Dear Customer,\n\nWe are giving away free holi...",dear customer giving away free holiday vacatio...


STEP 5: Clean Labels Properly

In [ ]:
# Clean labels without mapping since they are already numeric (0 and 1)
df = df.dropna(subset=['Spam Label'])
df['Spam Label'] = df['Spam Label'].astype(int)

print("Label distribution:")
print(df['Spam Label'].value_counts())

Label distribution:
Spam Label
1    1000
0     500
Name: count, dtype: int64


STEP 6: Remove Empty Text Rows

In [ ]:
df = df.dropna(subset=['clean_text'])
df = df[df['clean_text'].str.strip() != ""]

STEP 7: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Ensure we only keep rows with valid text and labels
df = df.dropna(subset=['clean_text', 'Spam Label'])
df = df[df['clean_text'].str.strip() != ""]
df = df.reset_index(drop=True)

X = df['clean_text']
y = df['Spam Label'].astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train/Test split done ✅")
print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train/Test split done ✅
Train size: 1200
Test size: 300


STEP 8: Train Spam Detection Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Vectorization
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train Model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')
print(classification_report(y_test, y_pred))

Accuracy: 1.00
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       200

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



STEP 9: Model Evaluation

In [ ]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       200

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



STEP 10: Emotion Detection Function

In [ ]:
def detect_emotion(text):
    analysis = TextBlob(text)
    polarity = analysis.sentiment.polarity

    if polarity > 0.5:
        return "Urgent/Positive"
    elif polarity < 0:
        return "Angry/Negative"
    else:
        return "Neutral"

STEP 11: Final Prediction Function

In [ ]:
def classify_email(email):
    # Preprocess
    clean = preprocess_text(email)

    # Vectorize
    vector = vectorizer.transform([clean])

    # Predict spam or ham
    prediction = model.predict(vector)[0]

    if prediction == 1:
        return "Spam Email ❌"
    else:
        emotion = detect_emotion(email)
        return f"Not Spam ✅ | Emotion: {emotion}"

STEP 12: Test Your Model

In [ ]:
# Try sample emails
emails = [
    "Congratulations! You won a lottery. Claim now!",
    "Please respond urgently regarding your account",
    "I am very disappointed with your service",
    "Meeting is scheduled tomorrow at 10 AM"
]

for e in emails:
    print("Email:", e)
    print("Result:", classify_email(e))
    print("-" * 50)

Email: Congratulations! You won a lottery. Claim now!
Result: Spam Email ❌
--------------------------------------------------
Email: Please respond urgently regarding your account
Result: Spam Email ❌
--------------------------------------------------
Email: I am very disappointed with your service
Result: Not Spam ✅ | Emotion: Angry/Negative
--------------------------------------------------
Email: Meeting is scheduled tomorrow at 10 AM
Result: Not Spam ✅ | Emotion: Neutral
--------------------------------------------------
